# 📖 Lab 2: Rate Limiting Algorithms

The heart of a rate limiter: the algorithm that decides allow or reject. Four main approaches, each with different trade-offs around accuracy, memory, and burst handling.

## The Four Algorithms

| # | Algorithm | Memory | Accuracy | Burst Control |
|---|-----------|--------|----------|---------------|
| 1 | Fixed Window Counter | 1 counter/client | ❌ Boundary spike | ❌ No |
| 2 | Sliding Window Log | 1 timestamp/request | ✅ Perfect | ✅ Yes |
| 3 | Sliding Window Counter | 2 counters/client | 🟡 Approximate | 🟡 Approximate |
| 4 | **Token Bucket** ✅ | 2 values/client | ✅ Precise | ✅ Natural bursts |

We'll implement all four in pure Python, then build the production version of Token Bucket with **Redis + Lua scripting** (atomic, race-condition free).

## Learning Objectives

- Implement and visualize all 4 algorithms
- See the fixed window boundary problem in action
- Understand why Token Bucket wins for most use cases
- Build an atomic Token Bucket in Redis with Lua scripting
- Simulate concurrent gateway instances and prove no race conditions

## 🛠️ Setup

```bash
cd system-designs/rate-limiter
docker-compose up -d
```

Select the **"Rate Limiter (Python)"** kernel.

In [ ]:
import time
import threading
import redis

redis_client = redis.Redis(host="localhost", port=6381, decode_responses=True)
redis_client.flushdb()
print(f"✅ Redis: {'connected' if redis_client.ping() else 'FAILED'}")

## Algorithm 1: Fixed Window Counter

Simplest approach. Divide time into fixed windows (e.g., 1-minute buckets), count requests per window. Reset at window boundary.

```
Window: 12:00:00-12:00:59    Window: 12:01:00-12:01:59
       ████████░░ (80/100)          ██░░░░░░░░ (20/100)

⚠️ Problem: 100 requests at 12:00:59 + 100 at 12:01:00 = 200 in 2 seconds!
```

In [ ]:
class FixedWindowCounter:
    def __init__(self, limit: int, window_seconds: int):
        self.limit = limit
        self.window_seconds = window_seconds
        self.counters: dict[str, tuple[int, float]] = {}  # client -> (count, window_start)

    def is_allowed(self, client_id: str, now: float = None) -> bool:
        now = now or time.time()
        window_start = now - (now % self.window_seconds)

        count, ws = self.counters.get(client_id, (0, 0))
        if ws != window_start:
            count, ws = 0, window_start

        if count >= self.limit:
            return False

        self.counters[client_id] = (count + 1, ws)
        return True


# Demo: normal usage
fw = FixedWindowCounter(limit=10, window_seconds=1)
now = 1000.0  # fixed time for reproducibility

allowed = sum(1 for _ in range(15) if fw.is_allowed("alice", now + _ * 0.05))
print(f"📊 Fixed Window: 15 requests in 1 window → {allowed} allowed, {15 - allowed} blocked")

# Demo: boundary spike problem
fw2 = FixedWindowCounter(limit=10, window_seconds=1)
# 10 requests at end of window 1
end_of_window = 1000.99
for i in range(10):
    fw2.is_allowed("alice", end_of_window)

# 10 requests at start of window 2 (just 0.02 seconds later!)
start_of_next = 1001.01
allowed_next = sum(1 for i in range(10) if fw2.is_allowed("alice", start_of_next))

print(f"\n🚨 Boundary spike problem:")
print(f"  10 requests at t=1000.99 → all allowed")
print(f"  10 requests at t=1001.01 → {allowed_next} allowed (new window!)")
print(f"  → 20 requests in 0.02 seconds! Limit was 10/second.")

## Algorithm 2: Sliding Window Log

Keep every request timestamp. On each new request, remove entries older than the window, then check if count exceeds limit. **Perfect accuracy**, but O(n) memory per client.

In [ ]:
class SlidingWindowLog:
    def __init__(self, limit: int, window_seconds: int):
        self.limit = limit
        self.window_seconds = window_seconds
        self.logs: dict[str, list[float]] = {}

    def is_allowed(self, client_id: str, now: float = None) -> bool:
        now = now or time.time()
        if client_id not in self.logs:
            self.logs[client_id] = []

        # Remove expired timestamps
        cutoff = now - self.window_seconds
        self.logs[client_id] = [t for t in self.logs[client_id] if t > cutoff]

        if len(self.logs[client_id]) >= self.limit:
            return False

        self.logs[client_id].append(now)
        return True


# No boundary spike with sliding window!
sw = SlidingWindowLog(limit=10, window_seconds=1)

for i in range(10):
    sw.is_allowed("alice", 1000.99)  # 10 at end of "window"

allowed_next = sum(1 for i in range(10) if sw.is_allowed("alice", 1001.01))

print(f"✅ Sliding Window Log — no boundary spike:")
print(f"  10 requests at t=1000.99 → all allowed")
print(f"  10 requests at t=1001.01 → {allowed_next} allowed")
print(f"  → Only {10 + allowed_next} total. The log sees ALL 10 recent requests.")

# Memory cost
print(f"\n  ⚠️  Memory: storing {len(sw.logs['alice'])} timestamps for Alice")
print(f"     At 1000 req/min × 100M users = 100 BILLION timestamps. Not scalable.")

## Algorithm 3: Sliding Window Counter

Clever hybrid. Keep counters for current + previous window. Estimate the sliding count by weighting the previous window based on how far into the current window we are.

```
prev_window: 80 requests    current_window: 30 requests    Position: 40% into current
Sliding estimate = 80 × (1 - 0.4) + 30 = 80 × 0.6 + 30 = 48 + 30 = 78
```

Only 2 counters per client — O(1) memory. But it's an approximation.

In [ ]:
class SlidingWindowCounter:
    def __init__(self, limit: int, window_seconds: int):
        self.limit = limit
        self.window_seconds = window_seconds
        # client -> {window_start: count}
        self.windows: dict[str, dict[float, int]] = {}

    def is_allowed(self, client_id: str, now: float = None) -> bool:
        now = now or time.time()
        current_window = now - (now % self.window_seconds)
        prev_window = current_window - self.window_seconds

        if client_id not in self.windows:
            self.windows[client_id] = {}

        prev_count = self.windows[client_id].get(prev_window, 0)
        curr_count = self.windows[client_id].get(current_window, 0)

        # Weight: how far we are into the current window
        elapsed = now - current_window
        weight = 1 - (elapsed / self.window_seconds)

        # Estimated count = weighted previous + all of current
        estimated = prev_count * weight + curr_count

        if estimated >= self.limit:
            return False

        self.windows[client_id][current_window] = curr_count + 1
        return True


swc = SlidingWindowCounter(limit=10, window_seconds=1)

# Same boundary test
for i in range(10):
    swc.is_allowed("alice", 1000.99)

allowed = sum(1 for i in range(10) if swc.is_allowed("alice", 1001.01))
print(f"📊 Sliding Window Counter — boundary test:")
print(f"  10 requests at t=1000.99 → all allowed")
print(f"  10 requests at t=1001.01 → {allowed} allowed")
print(f"  → Weighted estimate catches the spike! Only 2 counters in memory.")

## Algorithm 4: Token Bucket ✅ (Our Choice)

Each client has a bucket with a max capacity. Tokens refill at a steady rate. Each request takes one token. No tokens → rejected.

```
Bucket capacity: 10 tokens    Refill rate: 2 tokens/second

t=0:   [●●●●●●●●●●] 10 tokens (full)
t=0.1: [●●●●●●●○○○]  7 tokens (3 requests used 3 tokens)
t=0.5: [●●●●●●●●○○]  8 tokens (1 refilled after 0.5s)
t=1.0: [●●●●●●●●●●] 10 tokens (refilled to max)
```

**Natural burst support:** idle clients accumulate tokens → can burst up to capacity. Sustained rate is capped at refill rate.

In [ ]:
class TokenBucket:
    def __init__(self, capacity: int, refill_rate: float):
        self.capacity = capacity          # max tokens
        self.refill_rate = refill_rate    # tokens per second
        self.buckets: dict[str, tuple[float, float]] = {}  # client -> (tokens, last_refill)

    def is_allowed(self, client_id: str, now: float = None) -> dict:
        now = now or time.time()

        tokens, last_refill = self.buckets.get(client_id, (self.capacity, now))

        # Refill tokens based on elapsed time
        elapsed = now - last_refill
        tokens = min(self.capacity, tokens + elapsed * self.refill_rate)

        if tokens >= 1:
            tokens -= 1
            self.buckets[client_id] = (tokens, now)
            return {"allowed": True, "remaining": int(tokens)}
        else:
            self.buckets[client_id] = (tokens, now)
            return {"allowed": False, "remaining": 0}


# Demo: burst then sustained
tb = TokenBucket(capacity=10, refill_rate=2)  # 10 burst, 2/sec sustained
now = 1000.0

print("📊 Token Bucket — burst then sustained:\n")

# Burst: 12 rapid requests
print("  Burst phase (12 rapid requests):")
for i in range(12):
    result = tb.is_allowed("alice", now + i * 0.01)
    status = "✅" if result["allowed"] else "❌"
    if i < 3 or i >= 9:
        print(f"    Request {i+1:2d}: {status} (remaining: {result['remaining']})")
    elif i == 3:
        print(f"    ... (requests 4-9 allowed)")

# Wait 3 seconds — tokens refill
print(f"\n  ⏳ Wait 3 seconds... (refill 2/sec × 3s = 6 tokens)")
refill_time = now + 3.0
result = tb.is_allowed("alice", refill_time)
print(f"  Request after wait: {'✅' if result['allowed'] else '❌'} (remaining: {result['remaining']})")

print(f"\n  💡 Burst of 10 allowed (bucket capacity), then blocked.")
print(f"     After waiting, tokens refilled — new requests allowed.")

## 🔧 Production Token Bucket: Redis + Lua Script

In-memory works for a single process. But with multiple API Gateway instances, we need **centralized state in Redis**. The critical requirement: the read-calculate-update must be **atomic** to prevent race conditions.

**Redis Lua scripts** execute atomically — the entire script runs as a single operation. No other Redis command can interleave.

In [ ]:
TOKEN_BUCKET_LUA = """
local key = KEYS[1]
local capacity = tonumber(ARGV[1])
local refill_rate = tonumber(ARGV[2])
local now = tonumber(ARGV[3])
local ttl = tonumber(ARGV[4])

-- Get current state
local tokens = tonumber(redis.call('HGET', key, 'tokens') or capacity)
local last_refill = tonumber(redis.call('HGET', key, 'last_refill') or now)

-- Refill tokens based on elapsed time
local elapsed = now - last_refill
tokens = math.min(capacity, tokens + elapsed * refill_rate)

local allowed = 0
local remaining = math.floor(tokens)

if tokens >= 1 then
    tokens = tokens - 1
    allowed = 1
    remaining = math.floor(tokens)
end

-- Update state atomically
redis.call('HSET', key, 'tokens', tostring(tokens))
redis.call('HSET', key, 'last_refill', tostring(now))
redis.call('EXPIRE', key, ttl)

return {allowed, remaining}
"""

# Register the script
token_bucket_script = redis_client.register_script(TOKEN_BUCKET_LUA)

CAPACITY = 10
REFILL_RATE = 2  # tokens/sec
TTL = 3600       # auto-cleanup after 1 hour inactive


def check_rate_limit(client_id: str) -> dict:
    """Production rate limit check — atomic Redis Lua script."""
    key = f"rl:bucket:{client_id}"
    now = time.time()

    result = token_bucket_script(keys=[key], args=[CAPACITY, REFILL_RATE, now, TTL])
    return {
        "allowed": bool(result[0]),
        "remaining": result[1],
    }


# Test it
redis_client.flushdb()

print("🔧 Redis Token Bucket (Lua Script):\n")
print(f"  Capacity: {CAPACITY}, Refill: {REFILL_RATE}/sec\n")

for i in range(13):
    result = check_rate_limit("alice")
    status = "✅ ALLOW" if result["allowed"] else "❌ REJECT"
    print(f"  Request {i+1:2d}: {status} (remaining: {result['remaining']})")

# Check Redis state
state = redis_client.hgetall("rl:bucket:alice")
print(f"\n  📊 Redis state: {state}")
print(f"  TTL: {redis_client.ttl('rl:bucket:alice')}s")

## 🧪 Proving No Race Conditions

The key test: simulate multiple gateway instances hitting Redis concurrently for the same client. With the Lua script, only the correct number of requests should be allowed.

In [ ]:
# Simulate 5 gateway instances hitting Redis concurrently
redis_client.flushdb()

NUM_GATEWAYS = 5
REQUESTS_PER_GATEWAY = 10
results_lock = threading.Lock()
all_results = {"allowed": 0, "rejected": 0}


def gateway_worker(gateway_id: int):
    for i in range(REQUESTS_PER_GATEWAY):
        result = check_rate_limit("alice_concurrent")
        with results_lock:
            if result["allowed"]:
                all_results["allowed"] += 1
            else:
                all_results["rejected"] += 1


# Launch all gateways simultaneously
threads = []
for gw in range(NUM_GATEWAYS):
    t = threading.Thread(target=gateway_worker, args=(gw,))
    threads.append(t)

for t in threads:
    t.start()
for t in threads:
    t.join()

total = NUM_GATEWAYS * REQUESTS_PER_GATEWAY

print(f"🧪 Concurrent Race Condition Test:\n")
print(f"  Gateways: {NUM_GATEWAYS}")
print(f"  Requests per gateway: {REQUESTS_PER_GATEWAY}")
print(f"  Total requests: {total}")
print(f"  Bucket capacity: {CAPACITY}")
print(f"\n  Results:")
print(f"    Allowed:  {all_results['allowed']}")
print(f"    Rejected: {all_results['rejected']}")

if all_results["allowed"] <= CAPACITY:
    print(f"\n  ✅ No race condition! Exactly {all_results['allowed']} requests allowed (≤ {CAPACITY} capacity).")
    print(f"     Lua script ensured atomic read-calculate-update.")
else:
    print(f"\n  🚨 Race condition detected! More than {CAPACITY} requests were allowed.")

## 🧹 Cleanup

In [ ]:
redis_client.flushdb()
print("✅ Redis cleaned up.")

## ✅ Summary

### Algorithm Comparison

| Algorithm | Memory | Accuracy | Boundary Spike | Burst Support | Choose When |
|-----------|--------|----------|----------------|---------------|-------------|
| **Fixed Window** | ✅ O(1) | ❌ 2× spike at boundary | ❌ Yes | ❌ No | Simple, non-critical limits |
| **Sliding Log** | ❌ O(n) | ✅ Perfect | ✅ None | ✅ Precise | Small scale, exact counting |
| **Sliding Counter** | ✅ O(1) | 🟡 ~97% accurate | 🟡 Minimal | 🟡 Approximate | Good balance, don't need bursts |
| **Token Bucket** ✅ | ✅ O(1) | ✅ Precise | ✅ None | ✅ Natural | **Most use cases** — burst + sustained |

### Why Token Bucket Wins

- **2 values per client** — capacity + last_refill (O(1) memory)
- **Natural burst handling** — idle clients accumulate tokens, can burst up to capacity
- **Sustained rate control** — refill rate caps long-term throughput
- **Simple to implement** — especially with Redis Lua scripts

### Production Architecture

```
Gateway A ──┐
Gateway B ──┼──> Redis (Lua script: atomic read-calc-update)
Gateway C ──┘    {client:bucket → {tokens, last_refill}, EXPIRE 1h}
```

- All gateways share centralized Redis state
- Lua script prevents race conditions (atomic operation)
- `EXPIRE` auto-cleans inactive buckets

**Next up:** Lab 3 — HTTP 429 responses, rate limit headers, and the deep dives (scaling, failover)